### Data Processing Notebook

In [1]:
import os
from pathlib import Path
import gzip
import json
import xml.etree.ElementTree as ET
from datetime import datetime
import faiss
import numpy as np

In [12]:
from pathlib import Path

BASELINE_DIR = Path("/home/Jdalton/codespace/medrag/data/baseline")
UPDATE_DIR   = Path("/home/Jdalton/codespace/medrag/data/updates")
JSON_DIR     = Path("/home/Jdalton/codespace/medrag/data/parsed")

JSON_DIR.mkdir(parents=True, exist_ok=True)

baseline_files = list(BASELINE_DIR.glob("*.xml.gz"))
update_files   = list(UPDATE_DIR.glob("*.xml.gz"))

all_files = baseline_files + update_files

print("Baseline files:", len(baseline_files))
print("Update files:", len(update_files))
print("Total XML files:", len(all_files))


Baseline files: 1274
Update files: 418
Total XML files: 1692


### Snapshot Window

In [13]:
baseline_end = datetime(2025, 1, 8, 23, 10)
snapshot_end = datetime(2026, 1, 13, 14, 2)

### XML to JSON Parser

In [14]:
def parse_pubmed(xml_gz_path):
    import gzip
    import xml.etree.ElementTree as ET

    with gzip.open(xml_gz_path, "rb") as f:
        context = ET.iterparse(f, events=("end",))
        for _, elem in context:
            if elem.tag != "PubmedArticle":
                continue

            pmid = elem.findtext("./MedlineCitation/PMID")
            title = elem.findtext(".//ArticleTitle") or ""
            abstract = " ".join(
                t.text or "" for t in elem.findall(".//AbstractText")
            )

            if pmid and (title or abstract):
                yield {
                    "pmid": pmid,
                    "title": title,
                    "abstract": abstract
                }

            elem.clear()


### Process all files

In [15]:
from multiprocessing import Pool
import json

OUT_FILE = JSON_DIR / "pubmed_corpus.jsonl"

def process_one_file(xml_file):
    count = 0
    lines = []
    for article in parse_pubmed(xml_file):
        lines.append(json.dumps(article))
        count += 1
    return count, lines

with Pool(processes=8) as pool, open(OUT_FILE, "a") as out_f:
    for xml_file, (count, lines) in zip(
        all_files, pool.imap(process_one_file, all_files)
    ):
        for line in lines:
            out_f.write(line + "\n")
        print(f"Processed {xml_file.name}: {count} articles")


Processed pubmed25n1028.xml.gz: 29729 articles
Processed pubmed25n1061.xml.gz: 29890 articles
Processed pubmed25n0917.xml.gz: 29894 articles
Processed pubmed25n0906.xml.gz: 29851 articles
Processed pubmed25n0692.xml.gz: 30000 articles
Processed pubmed25n0812.xml.gz: 29999 articles
Processed pubmed25n0513.xml.gz: 30000 articles
Processed pubmed25n1110.xml.gz: 29916 articles
Processed pubmed25n0324.xml.gz: 30000 articles
Processed pubmed25n1121.xml.gz: 29936 articles
Processed pubmed25n0548.xml.gz: 29999 articles
Processed pubmed25n0186.xml.gz: 30000 articles
Processed pubmed25n0809.xml.gz: 29999 articles
Processed pubmed25n1035.xml.gz: 29906 articles
Processed pubmed25n1262.xml.gz: 29943 articles
Processed pubmed25n0045.xml.gz: 30000 articles
Processed pubmed25n0774.xml.gz: 30000 articles
Processed pubmed25n0621.xml.gz: 30000 articles
Processed pubmed25n0466.xml.gz: 30000 articles
Processed pubmed25n0718.xml.gz: 30000 articles
Processed pubmed25n0516.xml.gz: 30000 articles
Processed pub

In [17]:
import json

with open(OUT_FILE, "r") as f:
    first_line = f.readline()
    first_article = json.loads(first_line)
    
print(first_article)

{'pmid': '32259311', 'title': 'Atorvastatin attenuates vascular remodelling in spontaneously hypertensive rats via the protein kinase D/extracellular signal-regulated kinase 5 pathway.', 'abstract': 'The present study was conducted to determine whether atorvastatin reduces hypertension-induced vascular remodelling and whether its effects involve protein kinase D (PKD) and extracellular signal-regulated kinase 5 (ERK5). We used 16-week-old spontaneously hypertensive rats (SHRs) and age-matched Wistar-Kyoto (WKY) rats. The blood pressure and serum lipid concentration were measured. Changes in the vascular morphology and histology were examined using H&E, Masson'}
